# compile_from_dict.ipynb
***example of compiling a tensorflow-free model from a dictionary saved with tf_to_dict, using either numpy or jax***
___


## imports

In [1]:
import json
import numpy as np
import jax.numpy as jnp

from compile_from_dict import numpy_compile, jax_compile


## load in dictionary made using `tf_to_dict`

In [2]:
model_name = 'M4-test'
with open(f'models/{model_name}.json', 'r') as fp:
    model_dict = json.load(fp)

## numpy_compile
let's take a look at how we use numpy_compile to interpret the `model_dict` dictionary made using `tf_to_dict`

In [22]:
numpy_model = numpy_compile(model_dict)

done! `numpy_model` is now a an object containing a flow of functions written purely in numpy which represent a forwards pass through the network

the example network, `pitchfork`, takes a set of 5 inputs and predicts 3 outputs on one branch and 38 on the other - let's define a test point of arbitrary values, and check that this is happening:

In [23]:
numpy_single_point = np.array([[0.5,0.5,0.5,0.5,0.5,0.5,0.5]])

In [24]:
numpy_model.forward_pass(numpy_single_point)

array([[ 1.32138377, -1.68991076,  0.76779496,  3.47273024,  3.39449149]])

nice! that seemed fast, but let's time it:

In [25]:
%%time
numpy_model.forward_pass(numpy_single_point)

CPU times: user 34 μs, sys: 970 μs, total: 1 ms
Wall time: 829 μs


array([[ 1.32138377, -1.68991076,  0.76779496,  3.47273024,  3.39449149]])

we typically want neural networks to predict on huge batches at once rather than just single points, though.

let's check that this functionality isn't lost when compiling from our dict, and time:

In [28]:
numpy_many_points = np.full((100000,7), 0.5)

In [29]:
%%time
numpy_model.forward_pass(numpy_many_points)

CPU times: user 31.5 s, sys: 3.1 s, total: 34.6 s
Wall time: 1.07 s


array([[ 1.32138377, -1.68991076,  0.76779496,  3.47273024,  3.39449149],
       [ 1.32138377, -1.68991076,  0.76779496,  3.47273024,  3.39449149],
       [ 1.32138377, -1.68991076,  0.76779496,  3.47273024,  3.39449149],
       ...,
       [ 1.32138377, -1.68991076,  0.76779496,  3.47273024,  3.39449149],
       [ 1.32138377, -1.68991076,  0.76779496,  3.47273024,  3.39449149],
       [ 1.32138377, -1.68991076,  0.76779496,  3.47273024,  3.39449149]])

also pretty fast!

however, we can definitely make this faster by using jax (which utilises the GPU where possible), and even faster if we then JIT compile the jax predict function!

## jax_compile
this time, let's try the same model but compiled entirely in jax - same process as before:

In [3]:
jax_model = jax_compile(model_dict)

the `jax_model` object is written entirely in jax.numpy - so we want to be careful that we're only passing in jax objects otherwise we might be losing valuable time!

lets define some test points like before:

In [31]:
jax_single_point = jnp.array([[0.5,0.5,0.5,0.5,0.5,0.5,0.5]])
jax_many_points = jnp.full((100000,7), 0.5)

and then we can perform a forward pass and time for one point:

In [4]:
%%time
jax_model.forward_pass(jax_single_point)

NameError: name 'jax_single_point' is not defined

or for a batch of points:

In [33]:
%%time
jax_model.forward_pass(jax_many_points)

CPU times: user 181 ms, sys: 412 ms, total: 593 ms
Wall time: 68.6 ms


Array([[ 1.3213847 , -1.6899098 ,  0.767795  ,  3.4727275 ,  3.3944902 ],
       [ 1.3213847 , -1.6899098 ,  0.767795  ,  3.4727275 ,  3.3944902 ],
       [ 1.3213847 , -1.6899098 ,  0.767795  ,  3.4727275 ,  3.3944902 ],
       ...,
       [ 1.3213847 , -1.6899098 ,  0.767795  ,  3.4727275 ,  3.3944902 ],
       [ 1.3213849 , -1.6899096 ,  0.76779526,  3.4727278 ,  3.3944888 ],
       [ 1.3213849 , -1.6899096 ,  0.76779526,  3.4727278 ,  3.3944888 ]],      dtype=float32)

this may be faster than the numpy version or not depending on your machine.

one way that we can certainly speed this up is by jit compiling the flow of functions in jax_model:

## jax_compile.jit_forward_pass
as before, we compile our model from the dictionary:

In [5]:
jax_model = jax_compile(model_dict)

and define some jax friendly test points:

In [6]:
jax_single_point = jnp.array([[0.5,0.5,0.5,0.5,0.5,0.5,0.5]])
jax_many_points = jnp.full((100000,7), 0.5)

now let's try using the jit compiled version of `forward_pass`, and time:

In [7]:
%%time
jax_model.jit_forward_pass(jax_single_point)

CPU times: user 146 ms, sys: 11 ms, total: 157 ms
Wall time: 150 ms


Array([[ 1.32138377, -1.68991076,  0.76779496,  3.47273024,  3.39449149]],      dtype=float64)

oh dear! this is (probably) slower than your `numpy_model.forward_pass(np_single_point)` or `jax_model.forwad_pass(jax_single_point)` cells!

actually, this is entirely expected because of the way JIT compilation works - the flow of functions is compiled each time for a **specific input shape**, and there is a small overhead associated with JIT compilation.

***this is an important point - JIT compiling might not help us (and in fact slow things down) if our batch sizes change dynamically!***

let's see whether we do better now that we've compiled the single point pass:

In [37]:
%%time
jax_model.jit_forward_pass(jax_single_point)

CPU times: user 440 μs, sys: 62 μs, total: 502 μs
Wall time: 286 μs


Array([[ 1.3213847, -1.6899105,  0.7677949,  3.4727302,  3.3944914]],      dtype=float32)

nice!

what about for the batch of many points? we'll run and time one cell to compile:

In [38]:
%%time
jax_model.jit_forward_pass(jax_many_points)

CPU times: user 127 ms, sys: 9.57 ms, total: 136 ms
Wall time: 129 ms


Array([[ 1.3213847 , -1.6899098 ,  0.767795  ,  3.4727275 ,  3.3944902 ],
       [ 1.3213847 , -1.6899098 ,  0.767795  ,  3.4727275 ,  3.3944902 ],
       [ 1.3213847 , -1.6899098 ,  0.767795  ,  3.4727275 ,  3.3944902 ],
       ...,
       [ 1.3213847 , -1.6899098 ,  0.767795  ,  3.4727275 ,  3.3944902 ],
       [ 1.3213849 , -1.6899096 ,  0.76779526,  3.4727278 ,  3.3944888 ],
       [ 1.3213849 , -1.6899096 ,  0.76779526,  3.4727278 ,  3.3944888 ]],      dtype=float32)

and then again to time the compiled version:

In [39]:
%%time
jax_model.jit_forward_pass(jax_many_points)

CPU times: user 0 ns, sys: 125 ms, total: 125 ms
Wall time: 3.46 ms


Array([[ 1.3213847 , -1.6899098 ,  0.767795  ,  3.4727275 ,  3.3944902 ],
       [ 1.3213847 , -1.6899098 ,  0.767795  ,  3.4727275 ,  3.3944902 ],
       [ 1.3213847 , -1.6899098 ,  0.767795  ,  3.4727275 ,  3.3944902 ],
       ...,
       [ 1.3213847 , -1.6899098 ,  0.767795  ,  3.4727275 ,  3.3944902 ],
       [ 1.3213849 , -1.6899096 ,  0.76779526,  3.4727278 ,  3.3944888 ],
       [ 1.3213849 , -1.6899096 ,  0.76779526,  3.4727278 ,  3.3944888 ]],      dtype=float32)

this should be much faster!